In [40]:
# !git clone https://github.com/correlllab/magpie_control.git
# !pip install -e magpie_control
from magpie_control import realsense_wrapper as real
import numpy as np
from PIL import Image

devices = real.poll_devices()

rsc = real.RealSense(fps=15, w=640, h=480, device_name="D405")
rsc.initConnection(device_serial=devices['D405'])

workspace_rs = real.RealSense(zMax=5, fps=6, w=640, h=480, device_name="D435")
workspace_rs.initConnection(device_serial=devices['D435'])


In [41]:
# from magpie_perception.label_dino import LabelDINO
from magpie_perception.label_owlv2 import LabelOWLv2
label_vit = LabelOWLv2(topk=3)
label_vit.init()

In [45]:
import sys
# sys.path.append("../")
# from src.magpie_perception.mask_sam2 import MaskSAM2
from magpie_perception.mask_sam2 import MaskSAM2
mask_sam2 = MaskSAM2("facebook/sam2.1-hiera-large")

In [57]:
p, rgbd_image = rsc.getPCD()
image = np.array(rgbd_image.color)
image = Image.fromarray(image)
# this is taking the image


In [58]:
# this is running it through owl v2
import numpy as np
from PIL import Image
# image = Image.open("test.jpg").convert("RGB")
image = np.array(image)
label_vit.set_threshold(0.001)
queries = ["a red block", "a blue block", "yellow block"]
queries = ["bag handle"]
abbrevq = ["bag handle"]
results, boxes, scores, labels = label_vit.label(image, queries, abbrevq, plot=True, topk=True)



In [59]:
# plots the image and bounding boxes of whichever index we assigned 
# make sure to close the image 
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use('TkAgg')
label_vit.plot_predictions(index=2)
plt.show()

In [60]:
# gets the masks of the 5 highest confidence labels
mask_sam2.set_image_and_labels(np.array(rgbd_image.color), label_vit.sorted_boxes_coords[:5], label_vit.sorted_labels)
masks = mask_sam2.get_masks(labels)

In [65]:
# plots the 3 masks of the three highest confidence labels.
matplotlib.use('TkAgg')
mask_sam2.plot_image(np.array(rgbd_image.color), masks, label_vit.sorted_boxes_coords, label_vit.sorted_scores[:3])
plt.show()

In [66]:
# dump rgbd image, masks, boxes, scores, labels to disk
import time
t = time.time()
object_name = queries[0]
filepath = f"bag_handle_data/{object_name}_{t}"

# make filepath if it doesnt exist
import os
os.makedirs(filepath, exist_ok=True)

# save rgbd image
np.save(filepath + "/rgbd_color.npy", np.array(rgbd_image.color))
np.save(filepath + "/rgbd_depth.npy", np.array(rgbd_image.depth))
np.save(filepath + "/masks.npy", masks)
np.save(filepath + "/boxes.npy", boxes)
np.save(filepath + "/scores.npy", scores)
np.save(filepath + "/labels.npy", labels)
np.save(filepath + "/intrinsic_matrix.npy", rsc.pinholeInstrinsics.intrinsic_matrix)
np.save(filepath + "/extrinsic_matrix.npy", rsc.extrinsics)

In [67]:
import open3d as o3d

color = np.load(filepath + "/rgbd_color.npy", allow_pickle=True)
depth = np.load(filepath + "/rgbd_depth.npy", allow_pickle=True)

depth_o3d = o3d.geometry.Image((color).astype(np.uint8))
rgb_o3d = o3d.geometry.Image((color).astype(np.uint8))
new_rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(rgb_o3d, depth_o3d)

from magpie_perception import pcd
index = 0
rgbd_image, mcpcd, tmat, pca = pcd.get_segment(mask_sam2.masks.astype(bool), 
                                          index, 
                                          rgbd_image, 
                                        #   new_rgbd, 
                                          rsc, 
                                          type="mask", 
                                          viz_scale=2500.0, 
                                          display=True,
                                          method="iterative")

modified indices: [0 1 2]
z-axis dot product: [0.69884791]
[Open3D INFO] Window window_4 created.


WebVisualizer(window_uid='window_4')

[Open3D INFO] Sending init frames to window_4.


[1076:067][11063] (stun_port.cc:96): Binding request timed out from 192.168.0.x:48444 (enp3s0)
